# GISMO Fine-Tuning Notebook

Fine-tunes **Phi-3-mini-4k-instruct** on GISMO's training data using LoRA via Unsloth.
Exports a GGUF file you can load directly into Ollama on your local machine.

**Requirements:**
- Runtime → Change runtime type → **T4 GPU** (free tier is fine)
- ~15 minutes to train on 104 examples

**Workflow:**
1. Install dependencies
2. Upload `gismo_training_data.jsonl`
3. Load base model + configure LoRA
4. Train
5. Export to GGUF
6. Download and load into Ollama

## Step 1 — Install dependencies

Unsloth dramatically reduces VRAM usage and speeds up training. This takes ~3 minutes.

In [ ]:
%%capture
import subprocess, sys

# Install Unsloth (auto-detects CUDA version)
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "unsloth[colab-new]",
    "trl>=0.8.6",
    "transformers>=4.40",
    "datasets>=2.18",
    "accelerate>=0.29",
    "peft>=0.10",
    "bitsandbytes>=0.43",
], check=True)

print("Dependencies installed.")

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU found — go to Runtime > Change runtime type > T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Step 2 — Upload training data

Upload `data/training/gismo_training_data.jsonl` from the GISMO repo.

In [ ]:
from google.colab import files
import json, pathlib

uploaded = files.upload()  # select gismo_training_data.jsonl

data_path = list(uploaded.keys())[0]
records = [json.loads(line) for line in pathlib.Path(data_path).read_text().splitlines() if line.strip()]
print(f"Loaded {len(records)} training examples")
print("Sample:", json.dumps(records[0], indent=2))

## Step 3 — Load base model

Loads Phi-3-mini in 4-bit quantization (~2 GB VRAM) and attaches LoRA adapters.

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-3-mini-4k-instruct",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,          # auto-detect: float16 on T4
    load_in_4bit=True,
)

print("Base model loaded.")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                          # LoRA rank — higher = more capacity, more VRAM
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

model.print_trainable_parameters()

## Step 4 — Prepare dataset

Converts the `{"messages": [...]}` format into formatted text using the model's chat template.

In [ ]:
from datasets import Dataset

def format_example(record):
    """Apply the model's chat template to a messages record."""
    return tokenizer.apply_chat_template(
        record["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )

formatted = [{"text": format_example(r)} for r in records]
dataset = Dataset.from_list(formatted)

print(f"Dataset size: {len(dataset)}")
print("\nFormatted sample:")
print(dataset[0]["text"])

## Step 5 — Train

~15 minutes on a T4. Watch the loss drop — target below 0.5 by the final steps.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,   # effective batch = 8
        num_train_epochs=3,
        warmup_steps=10,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="/tmp/gismo-ft-checkpoints",
        report_to="none",
    ),
)

print("Starting training...")
trainer_stats = trainer.train()
print(f"\nTraining complete. Final loss: {trainer_stats.training_loss:.4f}")

## Step 6 — Export to GGUF

Merges LoRA weights into the base model and quantizes to `q4_k_m` — the best quality/size
tradeoff for Ollama on CPU/integrated GPU.

In [ ]:
import os

GGUF_DIR = "/tmp/gismo-gguf"
os.makedirs(GGUF_DIR, exist_ok=True)

print("Merging LoRA and exporting to GGUF (q4_k_m)...")
print("This takes ~5 minutes.")

model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method="q4_k_m",
)

gguf_files = [f for f in os.listdir(GGUF_DIR) if f.endswith(".gguf")]
print(f"\nGGUF files created: {gguf_files}")
for f in gguf_files:
    size_mb = os.path.getsize(os.path.join(GGUF_DIR, f)) / 1e6
    print(f"  {f}: {size_mb:.0f} MB")

## Step 7 — Quick sanity check

Ask the fine-tuned model who it is before downloading.

In [ ]:
FastLanguageModel.for_inference(model)

test_messages = [
    {"role": "user", "content": "Who are you?"},
]

inputs = tokenizer.apply_chat_template(
    test_messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=200,
    temperature=0.7,
    do_sample=True,
)

reply = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("GISMO says:")
print(reply)

## Step 8 — Download GGUF

Downloads the GGUF file to your local machine.

In [ ]:
from google.colab import files
import os

GGUF_DIR = "/tmp/gismo-gguf"
gguf_files = [f for f in os.listdir(GGUF_DIR) if f.endswith(".gguf")]

for fname in gguf_files:
    fpath = os.path.join(GGUF_DIR, fname)
    print(f"Downloading {fname} ({os.path.getsize(fpath)/1e6:.0f} MB)...")
    files.download(fpath)

## Step 9 — Load into Ollama on your local machine

Run these commands **on your Windows machine** after the download completes.

```bash
# 1. Move the GGUF into the repo
mv ~/Downloads/unsloth.Q4_K_M.gguf D:/repos/gismo/models/gismo-ft.Q4_K_M.gguf

# 2. Create a Modelfile pointing at the fine-tuned weights
#    (keeps all PARAMETER and SYSTEM settings from your existing Modelfile)
cat > D:/repos/gismo/Modelfile.ft << 'EOF'
FROM ./models/gismo-ft.Q4_K_M.gguf

PARAMETER temperature 0.7
PARAMETER top_p 0.9
PARAMETER top_k 40
PARAMETER repeat_penalty 1.1
PARAMETER num_ctx 4096
EOF

# 3. Register the model with Ollama
cd D:/repos/gismo
ollama create gismo -f Modelfile.ft

# 4. Test it
ollama run gismo "Who are you?"
```

---

### Tips for improving results

| Issue | Fix |
|---|---|
| Model forgets it's GISMO | Add more identity/persona examples to training data |
| Repeats itself | Increase `repeat_penalty` to 1.15 in Modelfile |
| Too verbose | Add concise-answer examples to training data |
| Loss stuck above 1.0 | Increase `num_train_epochs` to 5, lower `learning_rate` to 1e-4 |
| Want higher quality | Increase LoRA `r` to 32 (needs more VRAM — use Colab Pro) |

The web chat at `gismo/web/api.py` already uses `model="gismo"` so it will automatically pick up the re-registered model — no code changes needed.